In [3]:
#Amy Independent Research, Fall 2024
import networkx as nx
import osmnx as ox
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import ScrollZoomToggler
import matplotlib.pyplot as plt
import shapely
from glob import glob
from datetime import timedelta
import osmnx.settings as settings
import osmnx.features as features

ox.__version__

settings.cache_folder = "/tmp/cache"

For each model (Model 1 to Model 8), these are the regressors.
Dependent Variable: Average Daily Trip Counts (avgdtc)
1) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Intercept
2) Independent Variables:  Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Electric Bike Proportion (elecbikep), Intercept
3) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Electric Bike Proportion (elecbikep), Electric Bike Proportion * Temperature (elecbikep_avgt), Intercept
4) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Membership Proportion (memberp), Intercept
5) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Membership Proportion (memberp), Membership Proportion * Bike Lane Length (memberp_bl), Intercept
6) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Membership Proportion (memberp), Electric Bike Proportion (elecbikep), Electric Bike Proportion * Temperature (elecbikep_avgt), Membership Proportion * Bike Lane Length (memberp_bl), Membership Proportion * Electric Bike Proportion (memberp_elecbikep), Intercept
7) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Active Stations (activesta), Gravity Attractiveness, Intercept

avgdtc: average daily trip counts; avgpp: average precipitation (inch); avgsd: average snow depth (in); avgt: average temperature (F); avgws: average wind speed (mph); bl: total bike lane length (mile); activesta – average number of daily active stations; elecbikep - electric bike proportion; elecbikep_avgt - interaction term between electric bike proportion and average temperature (F); memberp - Membership Proportion; memberp_bl - interaction term between membership proportion and bike lane length (mile); memberp_elecbikep - interaction term between membership proportion and electric bike proportion

AR² refers to the autoregressive process where the current value of the dependent variable (Average Daily Trip Counts) depends on its lagged values up to the second order (2 previous time steps). It captures the effect of temporal autocorrelation in the dependent variable. For example, high trip counts on one day might be followed by high trip counts on the next day (due to momentum or recurring patterns).

ARCH³ refers to the third-order autoregressive conditional heteroskedasticity component of the model. ARCH³ models the conditional variance of the residuals (𝜎t squared) as a function of the squared residuals from up to 3 previous time steps. _cons (Intercept) represents the constant term in the model. It accounts for the baseline level of average daily trip counts when all other variables (independent variables and lagged terms) are zero.

The log-likelihood measures how well the model fits the data. Higher values indicate a better fit.

The Likelihood Ratio Test (LRT) compares the goodness-of-fit between two nested models (e.g., one model is a simplified version of the other). Tests whether adding parameters (e.g., additional lag terms or ARCH terms) significantly improves the model. Adding more lags or ARCH terms should increase the log-likelihood if they improve model fit. A significant p-value for the LRT indicates that the more complex model (with additional parameters) provides a significantly better fit.

Model 1: Observation on average over seven consecutive days in New York City. (ARCH)

Model 2: Observation on average over seven consecutive days in Non-Manhattan.

Model 3: Observation on average over seven consecutive days in Manhattan.

Model 4: Only average weekdays are included in each observation in Manhattan. 

Model 5: Only average weekends are included in each observation in Manhattan.

Model 6: Observation on average over seven consecutive days in Brooklyn.

Model 7: Only average weekdays are included in each observation in Brooklyn. 

Model 8: Only average weekends are included in each observation in Brooklyn.

# Clean Raw Bike Trip Data from CitiBike

In [2]:
file_path = "/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_1.csv"

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_39937/4193616194.py:4: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [5]:
# Define bounding boxes for each borough
borough_bounds = {
    'Manhattan': {'lat_min': 40.70, 'lat_max': 40.88, 'lng_min': -74.02, 'lng_max': -73.90},
    'Brooklyn': {'lat_min': 40.57, 'lat_max': 40.73, 'lng_min': -74.04, 'lng_max': -73.85},
    'Queens': {'lat_min': 40.54, 'lat_max': 40.80, 'lng_min': -73.95, 'lng_max': -73.70},
    'Bronx': {'lat_min': 40.79, 'lat_max': 40.91, 'lng_min': -73.93, 'lng_max': -73.80},
    'Staten Island': {'lat_min': 40.49, 'lat_max': 40.65, 'lng_min': -74.25, 'lng_max': -74.05}
}

def get_borough(lat, lng):
    for borough, bounds in borough_bounds.items():
        if bounds['lat_min'] <= lat <= bounds['lat_max'] and bounds['lng_min'] <= lng <= bounds['lng_max']:
            return borough
    return 'Other'

# Apply the get_borough function to determine the borough based on latitude and longitude
df['borough'] = df.apply(lambda row: get_borough(row['start_lat'], row['start_lng']), axis=1)

NameError: name 'df' is not defined

In [89]:
# Calculate weighted average helper function
def weighted_average(df, value_col, weight_col):
    """
    Calculate weighted average for a given column using weights.
    """
    return np.average(df[value_col], weights=df[weight_col])

# Parse date columns with single-digit handling
df['started_at'] = pd.to_datetime(df['started_at'], format='%m/%d/%Y', errors='coerce')
df['ended_at'] = pd.to_datetime(df['ended_at'], format='%m/%d/%Y', errors='coerce')

# Drop rows where parsing failed
df = df.dropna(subset=['started_at', 'ended_at'])

# Extract time-related features
df['year'] = df['started_at'].dt.year
df['month'] = df['started_at'].dt.month
df['week_number'] = df['started_at'].dt.isocalendar().week
df['day_of_week'] = df['started_at'].dt.weekday  # 0=Monday, 6=Sunday

# Calculate start and end dates for each week
df['start_date'] = pd.to_datetime(df['year'].astype(str) + "-01-01") + pd.to_timedelta((df['week_number'] - 1) * 7, unit='d')
df['end_date'] = df['start_date'] + pd.Timedelta(days=6)

# Recalculate the month based on start_date
df['month'] = df['start_date'].dt.month

# Filter for data from 2021 onward
df = df[df['year'] >= 2021]

# Calculate trip duration in minutes
df['trip_duration_min'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60

# Weekly aggregation at borough level
weekly_aggregated = df.groupby(['year', 'month', 'week_number', 'borough']).agg(
    total_trips=('ride_id', 'count'),
    weekday_trips=('day_of_week', lambda x: (x < 5).sum()),
    weekend_trips=('day_of_week', lambda x: (x >= 5).sum()),
    electric_bike_rides=('rideable_type', lambda x: (x == 'electric_bike').sum()),
    classic_bike_rides=('rideable_type', lambda x: (x == 'classic_bike').sum()),
    member_rides=('member_casual', lambda x: (x == 'member').sum()),
    casual_rides=('member_casual', lambda x: (x == 'casual').sum()),
    avg_trip_duration=('trip_duration_min', 'mean'),
    unique_start_stations=('start_station_id', 'nunique'),
    unique_end_stations=('end_station_id', 'nunique')
).reset_index()

# Calculate proportions
weekly_aggregated['electric_bike_proportion'] = weekly_aggregated['electric_bike_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['classic_bike_proportion'] = weekly_aggregated['classic_bike_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['member_proportion'] = weekly_aggregated['member_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['casual_proportion'] = weekly_aggregated['casual_rides'] / weekly_aggregated['total_trips']

# Calculate average daily, weekday, and weekend trips
weekly_aggregated['avg_daily_trips'] = weekly_aggregated['total_trips'] / 7
weekly_aggregated['avg_weekday_trips'] = weekly_aggregated['weekday_trips'] / 5
weekly_aggregated['avg_weekend_trips'] = weekly_aggregated['weekend_trips'] / 2

# NYC total
nyc_total = weekly_aggregated.groupby(['year', 'month', 'week_number']).agg(
    total_trips=('total_trips', 'sum'),
    weekday_trips=('weekday_trips', 'sum'),
    weekend_trips=('weekend_trips', 'sum'),
    electric_bike_rides=('electric_bike_rides', 'sum'),
    classic_bike_rides=('classic_bike_rides', 'sum'),
    member_rides=('member_rides', 'sum'),
    casual_rides=('casual_rides', 'sum'),
    unique_start_stations=('unique_start_stations', 'sum'),
    unique_end_stations=('unique_end_stations', 'sum'),
    avg_trip_duration=('avg_trip_duration', lambda x: weighted_average(weekly_aggregated, 'avg_trip_duration', 'total_trips'))
).reset_index()

# Add average trips for NYC Total
nyc_total['avg_daily_trips'] = nyc_total['total_trips'] / 7
nyc_total['avg_weekday_trips'] = nyc_total['weekday_trips'] / 5
nyc_total['avg_weekend_trips'] = nyc_total['weekend_trips'] / 2
nyc_total['borough'] = 'NYC Total'

# Non-Manhattan total
non_manhattan_total = weekly_aggregated[weekly_aggregated['borough'] != 'Manhattan'].groupby(['year', 'month', 'week_number']).agg(
    total_trips=('total_trips', 'sum'),
    weekday_trips=('weekday_trips', 'sum'),
    weekend_trips=('weekend_trips', 'sum'),
    electric_bike_rides=('electric_bike_rides', 'sum'),
    classic_bike_rides=('classic_bike_rides', 'sum'),
    member_rides=('member_rides', 'sum'),
    casual_rides=('casual_rides', 'sum'),
    unique_start_stations=('unique_start_stations', 'sum'),
    unique_end_stations=('unique_end_stations', 'sum'),
    avg_trip_duration=('avg_trip_duration', lambda x: weighted_average(weekly_aggregated[weekly_aggregated['borough'] != 'Manhattan'], 'avg_trip_duration', 'total_trips'))
).reset_index()

# Add average trips for Non-Manhattan Total
non_manhattan_total['avg_daily_trips'] = non_manhattan_total['total_trips'] / 7
non_manhattan_total['avg_weekday_trips'] = non_manhattan_total['weekday_trips'] / 5
non_manhattan_total['avg_weekend_trips'] = non_manhattan_total['weekend_trips'] / 2
non_manhattan_total['borough'] = 'Non-Manhattan Total'

# Combine with original data
updated_weekly_aggregated = pd.concat([weekly_aggregated, nyc_total, non_manhattan_total], ignore_index=True)

# Sort the DataFrame
updated_weekly_aggregated = updated_weekly_aggregated.sort_values(by=['year', 'month', 'week_number', 'borough']).reset_index(drop=True)

# Recalculate proportions for NYC Total and Non-Manhattan Total
for group in ['NYC Total', 'Non-Manhattan Total']:
    subset = updated_weekly_aggregated[updated_weekly_aggregated['borough'] == group]
    updated_weekly_aggregated.loc[subset.index, 'electric_bike_proportion'] = subset['electric_bike_rides'] / subset['total_trips']
    updated_weekly_aggregated.loc[subset.index, 'classic_bike_proportion'] = subset['classic_bike_rides'] / subset['total_trips']
    updated_weekly_aggregated.loc[subset.index, 'member_proportion'] = subset['member_rides'] / subset['total_trips']
    updated_weekly_aggregated.loc[subset.index, 'casual_proportion'] = subset['casual_rides'] / subset['total_trips']

# Display the updated DataFrame
print(updated_weekly_aggregated)


Empty DataFrame
Columns: [year, month, week_number, borough, total_trips, weekday_trips, weekend_trips, electric_bike_rides, classic_bike_rides, member_rides, casual_rides, avg_trip_duration, unique_start_stations, unique_end_stations, electric_bike_proportion, classic_bike_proportion, member_proportion, casual_proportion, avg_daily_trips, avg_weekday_trips, avg_weekend_trips]
Index: []

[0 rows x 21 columns]


In [74]:
from datetime import timedelta

# Ensure 'started_at' is in datetime format
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

# Extract time-related features
df['year'] = df['started_at'].dt.year
df['month'] = df['started_at'].dt.month
df['week_number'] = df['started_at'].dt.isocalendar().week
df['day_of_week'] = df['started_at'].dt.weekday  # 0=Monday, 6=Sunday

# Filter data from 2021 onwards
df = df[df['year'] >= 2021]

# Calculate trip duration in minutes
df['trip_duration_min'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60

# Aggregate weekly data
weekly_aggregated = df.groupby(['year', 'month', 'week_number', 'borough']).agg(
    total_trips=('ride_id', 'count'),
    weekday_trips=('day_of_week', lambda x: (x < 5).sum()),  # Weekdays (Mon-Fri)
    weekend_trips=('day_of_week', lambda x: (x >= 5).sum()),  # Weekends (Sat-Sun)
    electric_bike_rides=('rideable_type', lambda x: (x == 'electric_bike').sum()),
    classic_bike_rides=('rideable_type', lambda x: (x == 'classic_bike').sum()),
    member_rides=('member_casual', lambda x: (x == 'member').sum()),
    casual_rides=('member_casual', lambda x: (x == 'casual').sum()),
    avg_trip_duration=('trip_duration_min', 'mean'),  # Average trip duration
    unique_start_stations=('start_station_id', 'nunique'),
    unique_end_stations=('end_station_id', 'nunique')
).reset_index()

# Calculate proportions and averages
weekly_aggregated['electric_bike_proportion'] = weekly_aggregated['electric_bike_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['classic_bike_proportion'] = weekly_aggregated['classic_bike_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['member_proportion'] = weekly_aggregated['member_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['casual_proportion'] = weekly_aggregated['casual_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['avg_daily_trips'] = weekly_aggregated['total_trips'] / 7
weekly_aggregated['avg_weekday_trips'] = weekly_aggregated['weekday_trips'] / 5
weekly_aggregated['avg_weekend_trips'] = weekly_aggregated['weekend_trips'] / 2

# Fix weighted averages for NYC Total and Non-Manhattan Total
def weighted_average(data, value_column, weight_column):
    """Calculate weighted average."""
    return np.average(data[value_column], weights=data[weight_column])

# Calculate NYC Total
nyc_total = weekly_aggregated.groupby(['year', 'month', 'week_number']).agg(
    total_trips=('total_trips', 'sum'),
    weekday_trips=('weekday_trips', 'sum'),
    weekend_trips=('weekend_trips', 'sum'),
    electric_bike_rides=('electric_bike_rides', 'sum'),
    classic_bike_rides=('classic_bike_rides', 'sum'),
    member_rides=('member_rides', 'sum'),
    casual_rides=('casual_rides', 'sum'),
    unique_start_stations=('unique_start_stations', 'sum'),
    unique_end_stations=('unique_end_stations', 'sum')
).reset_index()

nyc_total['avg_trip_duration'] = nyc_total.apply(
    lambda row: weighted_average(weekly_aggregated, 'avg_trip_duration', 'total_trips'),
    axis=1
)
nyc_total['borough'] = 'NYC Total'

# Calculate Non-Manhattan Total
non_manhattan_total = weekly_aggregated[weekly_aggregated['borough'] != 'Manhattan'].groupby(['year', 'month', 'week_number']).agg(
    total_trips=('total_trips', 'sum'),
    weekday_trips=('weekday_trips', 'sum'),
    weekend_trips=('weekend_trips', 'sum'),
    electric_bike_rides=('electric_bike_rides', 'sum'),
    classic_bike_rides=('classic_bike_rides', 'sum'),
    member_rides=('member_rides', 'sum'),
    casual_rides=('casual_rides', 'sum'),
    unique_start_stations=('unique_start_stations', 'sum'),
    unique_end_stations=('unique_end_stations', 'sum')
).reset_index()

non_manhattan_total['avg_trip_duration'] = non_manhattan_total.apply(
    lambda row: weighted_average(weekly_aggregated[weekly_aggregated['borough'] != 'Manhattan'], 'avg_trip_duration', 'total_trips'),
    axis=1
)
non_manhattan_total['borough'] = 'Non-Manhattan Total'

# Combine the original data with NYC and Non-Manhattan totals
updated_weekly_aggregated = pd.concat([weekly_aggregated, nyc_total, non_manhattan_total], ignore_index=True)

# Sort the DataFrame
updated_weekly_aggregated = updated_weekly_aggregated.sort_values(by=['year', 'month', 'week_number', 'borough']).reset_index(drop=True)

# Display updated DataFrame
print(updated_weekly_aggregated)




/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_63486/881241992.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['started_at'] = pd.to_datetime(df['started_at'])
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_63486/881241992.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ended_at'] = pd.to_datetime(df['ended_at'])
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_63486/881241992.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

    year  month  week_number              borough  total_trips  weekday_trips  \
0   2021      1            1                Bronx          413            324   
1   2021      1            1             Brooklyn        29635          20775   
2   2021      1            1            Manhattan       205760         150601   
3   2021      1            1            NYC Total       235808         171700   
4   2021      1            1  Non-Manhattan Total        30048          21099   
5   2021      1            2                Bronx          484            346   
6   2021      1            2             Brooklyn        33540          23335   
7   2021      1            2            Manhattan       229788         166959   
8   2021      1            2            NYC Total       263812         190640   
9   2021      1            2  Non-Manhattan Total        34024          23681   
10  2021      1            3                Bronx          321            246   
11  2021      1            3

In [16]:
# Define bounding boxes for each borough
borough_bounds = {
    'Manhattan': {'lat_min': 40.70, 'lat_max': 40.88, 'lng_min': -74.02, 'lng_max': -73.90},
    'Brooklyn': {'lat_min': 40.57, 'lat_max': 40.73, 'lng_min': -74.04, 'lng_max': -73.85},
    'Queens': {'lat_min': 40.54, 'lat_max': 40.80, 'lng_min': -73.95, 'lng_max': -73.70},
    'Bronx': {'lat_min': 40.79, 'lat_max': 40.91, 'lng_min': -73.93, 'lng_max': -73.80},
    'Staten Island': {'lat_min': 40.49, 'lat_max': 40.65, 'lng_min': -74.25, 'lng_max': -74.05}
}

def get_borough(lat, lng):
    for borough, bounds in borough_bounds.items():
        if bounds['lat_min'] <= lat <= bounds['lat_max'] and bounds['lng_min'] <= lng <= bounds['lng_max']:
            return borough
    return 'Other'

# Apply the get_borough function to determine the borough based on latitude and longitude
df['borough'] = df.apply(lambda row: get_borough(row['start_lat'], row['start_lng']), axis=1)

In [17]:
# Convert 'started_at' to datetime if not already done
df['started_at'] = pd.to_datetime(df['started_at'])

# Extract year, month, week number, and day of week
df['year'] = df['started_at'].dt.year
df['month'] = df['started_at'].dt.month
df['week_number'] = df['started_at'].dt.isocalendar().week
df['day_of_week'] = df['started_at'].dt.weekday  # 0=Monday, 6=Sunday

# Calculate trip duration in seconds and convert to minutes
df['trip_duration'] = (pd.to_datetime(df['ended_at']) - pd.to_datetime(df['started_at'])).dt.total_seconds()
df['trip_duration_min'] = df['trip_duration'] / 60

# Average trip duration for all days
avg_trip_duration_all = df['trip_duration_min'].mean()

# Average trip duration for weekdays (Monday to Friday)
avg_trip_duration_weekday = df[df['started_at'].dt.weekday < 5]['trip_duration_min'].mean()

# Average trip duration for weekends (Saturday and Sunday)
avg_trip_duration_weekend = df[df['started_at'].dt.weekday >= 5]['trip_duration_min'].mean()

In [18]:
# Aggregate weekly data
weekly_aggregated = df.groupby(['year', 'week_number', 'month', 'borough']).agg(
    total_trips=('ride_id', 'count'),  # Total trips
    weekday_trips=('day_of_week', lambda x: (x < 5).sum()),  # Weekdays (Mon-Fri)
    weekend_trips=('day_of_week', lambda x: (x >= 5).sum()),  # Weekends (Sat-Sun)
    electric_bike_rides=('rideable_type', lambda x: (x == 'electric_bike').sum()),  # Electric bike trips
    classic_bike_rides=('rideable_type', lambda x: (x == 'classic_bike').sum()),  # Classic bike trips
    member_rides=('member_casual', lambda x: (x == 'member').sum()),  # Member trips
    casual_rides=('member_casual', lambda x: (x == 'casual').sum()),  # Casual trips
    avg_trip_duration=('trip_duration_min', 'mean'),  # Average trip duration (minutes)
    unique_start_stations=('start_station_id', 'nunique'),  # Unique start stations
    unique_end_stations=('end_station_id', 'nunique')  # Unique end stations
).reset_index()

# Calculate proportions and averages
weekly_aggregated['electric_bike_proportion'] = (
    weekly_aggregated['electric_bike_rides'] / weekly_aggregated['total_trips']
)
weekly_aggregated['classic_bike_proportion'] = (
    weekly_aggregated['classic_bike_rides'] / weekly_aggregated['total_trips']
)
weekly_aggregated['member_proportion'] = (
    weekly_aggregated['member_rides'] / weekly_aggregated['total_trips']
)
weekly_aggregated['casual_proportion'] = (
    weekly_aggregated['casual_rides'] / weekly_aggregated['total_trips']
)

# Average daily, weekday, and weekend trips
weekly_aggregated['avg_daily_trips'] = weekly_aggregated['total_trips'] / 7
weekly_aggregated['avg_weekday_trips'] = weekly_aggregated['weekday_trips'] / 5  # Average weekday trips
weekly_aggregated['avg_weekend_trips'] = weekly_aggregated['weekend_trips'] / 2  # Average weekend trips

# Average trip duration for all days
avg_trip_duration_all = df['trip_duration_min'].mean()
# Average trip duration for weekdays (Monday to Friday)
avg_trip_duration_weekday = df[df['started_at'].dt.weekday < 5]['trip_duration_min'].mean()
# Average trip duration for weekends (Saturday and Sunday)
avg_trip_duration_weekend = df[df['started_at'].dt.weekday >= 5]['trip_duration_min'].mean()


In [19]:
# Calculate NYC total (sum across all boroughs)
nyc_total = weekly_aggregated.groupby(['year', 'month', 'week_number']).agg(
    total_trips=('total_trips', 'sum'),
    weekday_trips=('weekday_trips', 'sum'),
    weekend_trips=('weekend_trips', 'sum'),
    electric_bike_rides=('electric_bike_rides', 'sum'),
    classic_bike_rides=('classic_bike_rides', 'sum'),
    member_rides=('member_rides', 'sum'),
    casual_rides=('casual_rides', 'sum'),
    avg_trip_duration=('avg_trip_duration', lambda x: np.average(x, weights=weekly_aggregated.loc[x.index, 'total_trips'])),  # Weighted average
    unique_start_stations=('unique_start_stations', 'sum'),
    unique_end_stations=('unique_end_stations', 'sum')
).reset_index()

# Add a 'borough' column for NYC total
nyc_total['borough'] = 'NYC Total'

# Calculate Non-Manhattan total (sum across all boroughs except Manhattan)
non_manhattan_total = weekly_aggregated[weekly_aggregated['borough'] != 'Manhattan'].groupby(['year', 'month', 'week_number']).agg(
    total_trips=('total_trips', 'sum'),
    weekday_trips=('weekday_trips', 'sum'),
    weekend_trips=('weekend_trips', 'sum'),
    electric_bike_rides=('electric_bike_rides', 'sum'),
    classic_bike_rides=('classic_bike_rides', 'sum'),
    member_rides=('member_rides', 'sum'),
    casual_rides=('casual_rides', 'sum'),
    avg_trip_duration=('avg_trip_duration', lambda x: np.average(x, weights=weekly_aggregated.loc[x.index, 'total_trips'])),  # Weighted average
    unique_start_stations=('unique_start_stations', 'sum'),
    unique_end_stations=('unique_end_stations', 'sum')
).reset_index()

# Add a 'borough' column for Non-Manhattan total
non_manhattan_total['borough'] = 'Non-Manhattan Total'

# Combine the original data with the new NYC Total and Non-Manhattan Total rows
updated_weekly_aggregated = pd.concat([weekly_aggregated, nyc_total, non_manhattan_total], ignore_index=True)

# Sort the DataFrame by year, month, week_number, and borough for clarity
updated_weekly_aggregated = updated_weekly_aggregated.sort_values(by=['year', 'month', 'week_number', 'borough']).reset_index(drop=True)

# Recalculate proportions for NYC Total and Non-Manhattan Total
for group in ['NYC Total', 'Non-Manhattan Total']:
    subset = updated_weekly_aggregated[updated_weekly_aggregated['borough'] == group]
    updated_weekly_aggregated.loc[subset.index, 'electric_bike_proportion'] = (
        subset['electric_bike_rides'] / subset['total_trips']
    )
    updated_weekly_aggregated.loc[subset.index, 'classic_bike_proportion'] = (
        subset['classic_bike_rides'] / subset['total_trips']
    )
    updated_weekly_aggregated.loc[subset.index, 'member_proportion'] = (
        subset['member_rides'] / subset['total_trips']
    )
    updated_weekly_aggregated.loc[subset.index, 'casual_proportion'] = (
        subset['casual_rides'] / subset['total_trips']
    )

# Display the updated DataFrame
print(updated_weekly_aggregated)


Empty DataFrame
Columns: [year, week_number, month, borough, total_trips, weekday_trips, weekend_trips, electric_bike_rides, classic_bike_rides, member_rides, casual_rides, avg_trip_duration, unique_start_stations, unique_end_stations, electric_bike_proportion, classic_bike_proportion, member_proportion, casual_proportion, avg_daily_trips, avg_weekday_trips, avg_weekend_trips]
Index: []

[0 rows x 21 columns]


In [70]:
# Define the desired column order
column_order = [
    'week_label','year', 'month', 'week_number', 'borough', 
    'total_trips', 'avg_daily_trips', 'weekday_trips', 'avg_weekday_trips', 'weekend_trips', 'avg_weekend_trips',
    'electric_bike_rides', 'classic_bike_rides', 'electric_bike_proportion', 'classic_bike_proportion', 'member_rides', 'casual_rides',
    'member_proportion', 'casual_proportion', 'avg_trip_duration', 
    'unique_start_stations', 'unique_end_stations'
]

# Reorder the columns in the DataFrame
updated_weekly_aggregated = updated_weekly_aggregated[column_order]

# Display the reordered DataFrame
print(updated_weekly_aggregated.head())


           week_label  year  month  week_number              borough  \
0  Week 1 - 01 - 2021  2021      1            1                Bronx   
1  Week 1 - 01 - 2021  2021      1            1             Brooklyn   
2  Week 1 - 01 - 2021  2021      1            1            Manhattan   
3                 NaN  2021      1            1            NYC Total   
4                 NaN  2021      1            1  Non-Manhattan Total   

   total_trips  avg_daily_trips  weekday_trips  avg_weekday_trips  \
0          413        59.000000            324               64.8   
1        29635      4233.571429          20775             4155.0   
2       205760     29394.285714         150601            30120.2   
3       235808              NaN         171700                NaN   
4        30048              NaN          21099                NaN   

   weekend_trips  ...  classic_bike_rides  electric_bike_proportion  \
0             89  ...                 117                  0.716707   
1         

# Create a Loop to Sort All Processed Bike Trip Data into One Dataframe

In [21]:
# Create a new DataFrame for NYC-wide aggregation (excluding 'borough')
nyc_weekly_aggregated = df.groupby(['year', 'week_number']).agg(
    total_trips=('ride_id', 'count'),  # Total trips
    weekday_trips=('ride_id', lambda x: x[df.loc[x.index, 'day_of_week'] < 5].count()),  # Weekdays (Mon-Fri)
    weekend_trips=('ride_id', lambda x: x[df.loc[x.index, 'day_of_week'] >= 5].count()),  # Weekends (Sat-Sun)
    electric_bike_rides=('rideable_type', lambda x: (x == 'electric_bike').sum()),  # Electric bike trips
    classic_bike_rides=('rideable_type', lambda x: (x == 'classic_bike').sum()),  # Classic bike trips
    member_rides=('member_casual', lambda x: (x == 'member').sum()),  # Member trips
    casual_rides=('member_casual', lambda x: (x == 'casual').sum()),  # Casual trips
    avg_trip_duration=('trip_duration', 'mean'),  # Average trip duration
    unique_start_stations=('start_station_id', pd.Series.nunique),  # Unique start stations
    unique_end_stations=('end_station_id', pd.Series.nunique)  # Unique end stations
).reset_index()

# Add proportions and averages
nyc_weekly_aggregated['electric_bike_proportion'] = nyc_weekly_aggregated['electric_bike_rides'] / nyc_weekly_aggregated['total_trips']
nyc_weekly_aggregated['classic_bike_proportion'] = nyc_weekly_aggregated['classic_bike_rides'] / nyc_weekly_aggregated['total_trips']
nyc_weekly_aggregated['member_proportion'] = nyc_weekly_aggregated['member_rides'] / nyc_weekly_aggregated['total_trips']
nyc_weekly_aggregated['casual_proportion'] = nyc_weekly_aggregated['casual_rides'] / nyc_weekly_aggregated['total_trips']
nyc_weekly_aggregated['avg_daily_trips'] = nyc_weekly_aggregated['total_trips'] / 7
nyc_weekly_aggregated['avg_weekday_trips'] = nyc_weekly_aggregated['weekday_trips'] / 5
nyc_weekly_aggregated['avg_weekend_trips'] = nyc_weekly_aggregated['weekend_trips'] / 2

# Add start_date and end_date for each week
nyc_weekly_aggregated['start_date'] = nyc_weekly_aggregated.apply(
    lambda row: pd.to_datetime(f"{row['year']}-W{row['week_number']}-1", format="%Y-W%U-%w"), axis=1
)
nyc_weekly_aggregated['end_date'] = nyc_weekly_aggregated['start_date'] + timedelta(days=6)

# Save the NYC-wide aggregated DataFrame as a separate variable
nyc_df = nyc_weekly_aggregated.copy()

# Display the NYC-wide DataFrame
print(nyc_df.head())

KeyError: 'year'